# DATA103 Phase 1: Problem Formulation and Exploratory Data Analysis
## Automated American Sign Language (ASL) Fingerspelling Recognition via Hand Gesture Classification

* **Course:** DATA103 - Introduction to Machine Learning
* **Term / Academic Year:** Term 1, AY 2026-2027
* **Department:** Software Technology, College of Computer Studies, De La Salle University
* **Instructor:** Dr. John Jethro Virtusio
* **Authors:** Conda, Gopico, Marqueses, Piczon
* **Dataset Source:** [Sign Language MNIST on Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)

## 1. Background of the Study & Motivation

### 1.1 Problem Context & Significance
Over 70 million individuals globally rely on sign language as their primary mode of communication, yet fewer than 1% of non-deaf individuals are proficient in sign interpretation. This communication divide creates severe accessibility barriers in critical environments such as healthcare institutions, public customer service desks, emergency services, and educational settings.

While fluent sign language includes two-handed continuous gestures, body posture, and facial expressions, fingerspelling (spelling out words letter-by-letter using static single-hand configurations) represents the foundational building block for spelling personal names, technical terminology, and unstandardized concepts. Developing an automated, lightweight fingerspelling classifier can enable real-time transcription on resource-constrained consumer devices without requiring specialized hardware such as depth cameras or sensor gloves.

### 1.2 Research Objectives & Questions
This study investigates the performance of classical machine learning algorithms from the DATA103 curriculum on image-based gesture recognition:
1. **Linear vs. Non-linear Separability:** Can linear decision boundaries (Regularized Logistic Regression) adequately separate high-dimensional (784 features) flattened pixel representations, or are non-linear models (Decision Trees, Random Forests, Multi-Layer Perceptrons) fundamentally required to resolve visual ambiguities?
2. **Impact of Morphological Similarity:** Which specific gesture pairs (e.g., gestures with closed fists such as 'A', 'E', 'M', 'N', 'S') exhibit the highest confusion across classical models, and can distance-based neighborhood models (KNN) mitigate these classification errors?

### 1.3 Hypotheses
* **Primary Hypothesis:** Non-linear ensemble methods (Random Forest) and Multi-Layer Perceptrons (MLP) will achieve statistically superior Macro F1-scores compared to baseline linear models (Multinomial Logistic Regression), as pixel-level interactions along hand silhouettes are non-linear.
* **Secondary Hypothesis:** Linear models will struggle primarily with visually contiguous classes whose hand contours differ only by subtle thumb-over-finger placements.

In [1]:
# Environment setup and library imports
import string
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Visual formatting configurations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Core libraries imported successfully.")

Core libraries imported successfully.


In [3]:
# Ingest training and test datasets from the local data directory
train_path = "data/sign_mnist_train.csv"
test_path = "data/sign_mnist_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Training dataset shape: {train_df.shape} (rows, columns)")
print(f"Testing dataset shape:  {test_df.shape} (rows, columns)")

Training dataset shape: (27455, 785) (rows, columns)
Testing dataset shape:  (7172, 785) (rows, columns)


## 2. Dataset Structure and Data Integrity Verification

The dataset follows the standard MNIST benchmark structure:
* Each row represents an individual, pre-centered, 28x28 grayscale image of an American Sign Language hand gesture.
* **Column 0 (`label`):** The numerical target identifier representing the letter.
* **Columns 1 to 784 (`pixel1` to `pixel784`):** Raw grayscale brightness values ranging from 0 (pure black) to 255 (pure white).

In [ ]:
# Verify missing values and feature value ranges
train_nulls = train_df.isnull().sum().sum()
test_nulls = test_df.isnull().sum().sum()

pixel_min = train_df.iloc[:, 1:].min().min()
pixel_max = train_df.iloc[:, 1:].max().max()

print(f"Total null/missing values in Training Set: {train_nulls}")
print(f"Total null/missing values in Testing Set:  {test_nulls}")
print(f"Observed pixel intensity range: [{pixel_min}, {pixel_max}]")

# Preview initial records and columns
train_df.iloc[:5, :8]

## 3. Label Architecture & Class Mapping

The dataset contains **24 distinct classes** from the 26 letters of the English alphabet:
* **Excluded Classes:** The letters **'J'** (index 9) and **'Z'** (index 25) are omitted from the dataset because their production in American Sign Language requires continuous hand trajectory/motion paths, making them unsuitable for static 2D image classification.
* Labels represent class indices: 0 = 'A', 1 = 'B', ..., up to 24 = 'Y'.

In [ ]:
# Map numerical indices to alphabetic characters (skipping J and Z) and check class balance
alphabet = list(string.ascii_uppercase)
valid_letters = [char for char in alphabet if char not in ["J", "Z"]]
unique_labels = sorted(train_df["label"].unique())

label_to_letter = {lbl: valid_letters[i] for i, lbl in enumerate(unique_labels)}

plt.figure(figsize=(12, 4.5))
ax = sns.countplot(x="label", data=train_df, color="teal", edgecolor="black", linewidth=0.5)
plt.title("Training Set Class Distribution Across 24 Sign Gestures", fontsize=13, weight="bold")
plt.xlabel("Gesture Letter Class", fontsize=11)
plt.ylabel("Sample Count", fontsize=11)
plt.xticks(ticks=range(len(unique_labels)), labels=[label_to_letter[lbl] for lbl in unique_labels])
plt.tight_layout()
plt.show()

min_count = train_df["label"].value_counts().min()
max_count = train_df["label"].value_counts().max()
print(f"Class frequency range: Min = {min_count} samples, Max = {max_count} samples.")
print("Observation: The training data demonstrates a well-balanced distribution across all 24 classes.")

## 4. Visual Exploratory Data Analysis (EDA)

Because each row represents a flattened 784-dimensional vector (28x28 image), we reshape sample rows back into 2D matrices to visually inspect hand morphology, boundary quality, and spatial centering.

In [ ]:
# Reshape and plot one sample sign for each of the 24 classes
X_train_raw = train_df.drop("label", axis=1).values
y_train_raw = train_df["label"].values

fig, axes = plt.subplots(4, 6, figsize=(12, 8))
axes = axes.flatten()

for i, lbl in enumerate(unique_labels):
    sample_idx = np.where(y_train_raw == lbl)[0][0]
    img_matrix = X_train_raw[sample_idx].reshape(28, 28)
    
    axes[i].imshow(img_matrix, cmap="gray")
    axes[i].set_title(f"Class {lbl}: '{label_to_letter[lbl]}'", fontsize=10, weight="bold")
    axes[i].axis("off")

plt.suptitle("Visual Representation of 24 ASL Alphabet Signs (Reshaped 28x28)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Compute and display aggregate spatial density heatmap
mean_hand_matrix = X_train_raw.mean(axis=0).reshape(28, 28)

plt.figure(figsize=(5.5, 4.5))
plt.imshow(mean_hand_matrix, cmap="inferno")
plt.colorbar(label="Mean Intensity (0-255)")
plt.title("Aggregate Mean Pixel Intensity (Spatial Heatmap)", fontsize=12, weight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Observation: Spatial density shows consistent centering of hand silhouettes across the 28x28 bounding boxes.")

## 5. Planned Phase 2 Methodology

Following Phase 1 problem formulation and data validation, the Phase 2 implementation will follow this structured pipeline:

### 5.1 Preprocessing & Data Transformation
1. **Min-Max Normalization:** Scale all 784 pixel inputs from the raw [0, 255] integer interval to floating-point values in [0.0, 1.0]. This is critical to ensure distance metrics in KNN and gradient updates in Logistic Regression/MLP remain stable and well-conditioned.
2. **Data Partitioning:** Retain the official `sign_mnist_test.csv` (7,172 samples) strictly for final evaluation. Extract a 15% stratified validation subset from `sign_mnist_train.csv` for hyperparameter tuning and cross-validation.

### 5.2 Algorithm Selection from DATA103 Syllabus
We will implement and benchmark candidate models covered in the course curriculum:
* **K-Nearest Neighbors (KNN Classification - Week 4):** Non-parametric baseline using Euclidean distance across flattened 784-dimensional space.
* **Regularized Multinomial Logistic Regression (Weeks 5–6):** Linear baseline utilizing L1 (Lasso) and L2 (Ridge) penalty terms to prevent overfitting across the 784 collinear pixel features.
* **Decision Trees & Random Forest Classifier (Weeks 7–8):** Non-linear axis-aligned partitioning and bagging ensembles to assess feature importance across pixel coordinates.
* **Multi-Layer Perceptron (MLP - Week 11):** Fully connected feedforward neural network baseline utilizing backpropagation.
* **Exploratory Offshoot (Phase 2 Extension):** In accordance with project guidelines encouraging offshoots once the baseline is complete, we plan to train a Convolutional Neural Network (CNN) to evaluate how spatial kernel convolutions compare against tabular flattening.

### 5.3 Model Evaluation Metrics
* **Macro F1-Score & Accuracy:** Balanced evaluation across all 24 classes.
* **Confusion Matrix Diagnostic:** Systematic error analysis to measure inter-class confusion between morphologically similar hand signs (e.g., 'E' vs. 'S').